# Exercise 11: Audio analysis

Text mining needs... text. But a lot of data comes as **audio**: interviews, speeches, podcasts, meetings. In this exercise, we turn audio into text, then use a **large language model (LLM)** to clean it and analyse it.

| Part | What we do | Tool |
|---|---|---|
| 11a | Transcribe a speech in French | Whisper (OpenAI API) |
| 11b | Transcribe a speech in English | Whisper |
| 11c | Transcribe an interview, clean it with an LLM and save it | Whisper + GPT |
| 11d | Automate: extract the themes of several audio files with one function | Whisper + GPT |
| 11e | Improve a transcription with a stricter prompt | GPT |

The audio files are stored in `data/2_text_mining/exercice_11/`:
- `melenchon_campagne.mp3`: an extract of a campaign speech of Jean-Luc Mélenchon (presidential campaign 2022, [source](https://www.youtube.com/watch?v=x77tDojxgFU)),
- `trump_sotu.mp3`: an extract of the State of the Union address of Donald Trump (2018, [source](https://www.youtube.com/watch?v=jt1gBx4Fu5U)),
- `interview.mp3`: an extract of a TV interview of the sociologist Pierre Bourdieu ([source](https://www.youtube.com/watch?v=um7yAToVwcY)).

⚠️ **You need an OpenAI API key** ([platform.openai.com](https://platform.openai.com/api-keys)). Each call is billed (a few cents for the whole notebook). **Never write your key in the notebook**, and never commit it to git: the cell below asks for it, without displaying it.

### How to work

- Run the cells **one by one, from top to bottom** (`Shift + Enter`).
- ✏️ marks a cell you have to complete: replace the lines marked `## @COMPLETE`.
- 🧪 marks a test cell: run it to check your answer before moving on.
- 💡 gives you a hint, 🤔 asks you a question to think about.

In [ ]:
from getpass import getpass
from pathlib import Path

from openai import OpenAI

DATA_PATH = Path("../data/2_text_mining/exercice_11")

client = OpenAI(api_key=getpass("Paste your OpenAI API key: "))

## Exercise 11a: Transcribe a speech in French

**Whisper** is a speech recognition model: it listens to an audio file and returns its transcription. We send the file to the OpenAI API, which answers with the text.

The function below opens an audio file (in binary mode, `"rb"`) and sends it to Whisper. We will use it for every file of this notebook.

In [ ]:
def transcribe(filename):
    """Send an audio file to Whisper and return its transcription."""
    with open(DATA_PATH / filename, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(model="whisper-1", file=audio_file)
    return transcript.text

In [ ]:
transcript_fr = transcribe("melenchon_campagne.mp3")

print("--- Transcription (French) ---")
print(transcript_fr)

## Exercise 11b: Transcribe a speech in English

Whisper is **multilingual**: it detects the language by itself.

✏️ **Transcribe `trump_sotu.mp3`.**

In [ ]:
## @COMPLETE : transcribe the file trump_sotu.mp3
transcript_en = None

print("--- Transcription (English) ---")
print(transcript_en)

## Exercise 11c: Transcribe and clean an interview

The analysis of interviews is at the heart of many qualitative studies. Let's transcribe the interview of Pierre Bourdieu.

In [ ]:
transcript_interview = transcribe("interview.mp3")

print("--- Raw transcription (interview) ---")
print(transcript_interview)

### Clean the transcription with an LLM

A raw transcription contains many oral hesitations ("euh", repetitions) and no separation between the speakers. We ask an **LLM** (`gpt-4o-mini`) to clean it.

We talk to the LLM with two messages:
- a **system prompt**: the instructions, the role the model must play,
- a **user message**: the text to process.

The `temperature` controls the creativity of the model: low (close to 0) means more faithful and predictable answers, which is what we want here.

In [ ]:
system_prompt = """
Tu es sociologue.
Voici une transcription brute d'un entretien qualitatif.
Ton rôle est de :
1. Nettoyer les hésitations (les "euh", les mots répétés) pour rendre la lecture fluide.
2. Ajouter des sauts de ligne entre chaque prise de parole.
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Voici la retranscription brute :\n\n{transcript_interview}"},
    ],
    temperature=0.2,
)

clean_interview = response.choices[0].message.content
print("--- Cleaned interview ---")
print(clean_interview)

### Save the transcription

To keep the text, or to import it into a qualitative analysis software (NVivo, IRAMUTEQ, ...), we save it in a simple `.txt` file.

In [ ]:
with open("interview_bourdieu_clean.txt", "w", encoding="utf-8") as file:
    file.write(clean_interview)

print("Saved in interview_bourdieu_clean.txt")

## Exercise 11d: Automate (Don't Repeat Yourself!)

In a real study, we often have dozens of audio files. Rather than repeating the same code for each of them, we write **one reusable function** that does all the work (transcription, then extraction of the main themes), and call it in a loop.

✏️ **Complete the function `analyze_themes`**: transcribe the file, then send the transcription to the LLM with the system prompt `system_prompt_themes`.

💡 *Hint*: reuse the function `transcribe`, and copy the call to `client.chat.completions.create` of part 11c.

In [ ]:
system_prompt_themes = """
Tu es un chercheur en sciences sociales.
Identifie les 2 thèmes principaux abordés dans ce court extrait.
Pour chaque thème, donne simplement un titre court et une phrase d'explication.
"""


def analyze_themes(filename):
    """Transcribe an audio file with Whisper, then extract its main themes with an LLM."""
    print(f"\n========== {filename} ==========")

    ## @COMPLETE : transcribe the audio file
    raw_text = None
    print(f"[+] Transcription done ({len(raw_text)} characters)")

    ## @COMPLETE : ask the LLM to extract the themes of raw_text, with system_prompt_themes
    response = None

    return response.choices[0].message.content

Now, one loop processes the whole corpus:

In [ ]:
audio_files = ["trump_sotu.mp3", "melenchon_campagne.mp3", "interview.mp3"]

for filename in audio_files:
    print(analyze_themes(filename))

🤔 **Question**: how could you add a parameter to `analyze_themes` to change the system prompt, for instance to extract 3 themes, or to get the answer in English? Try it!

## Exercise 11e: Improve a transcription

Whisper performs well, but a raw transcription often lacks expressive punctuation, contains stutters, and may misspell proper nouns or acronyms (frequent in political speeches).

We now build an **expert transcriber** with a very strict system prompt: it must clean the speech of Jean-Luc Mélenchon (part 11a) while **keeping exactly** the style and the meaning of the speaker.

In [ ]:
expert_prompt = """
Tu es un transcripteur professionnel spécialisé en sciences sociales et en discours politiques.
Ta tâche est de corriger une transcription audio brute générée par une IA.

Instructions strictes :
1. Corrige les fautes d'orthographe, de grammaire et de ponctuation induites par l'IA (homophones, erreurs de découpage de phrases).
2. Supprime les bégaiements excessifs et les tics de langage inutiles ("euh", "ben", "alors voilà"), SAUF si cela modifie le sens ou l'intention de l'interlocuteur.
3. Rétablis les noms propres, acronymes ou termes techniques qui auraient pu être mal compris phonétiquement.
4. Ajoute une ponctuation expressive qui reflète le rythme et le ton (points d'exclamation, interrogations, pauses avec des virgules).
5. Ne modifie JAMAIS le vocabulaire choisi, garde le style et l'authenticité exacts de l'orateur.
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": expert_prompt},
        {"role": "user", "content": f"Voici la transcription brute à corriger :\n\n{transcript_fr}"},
    ],
    temperature=0.2,  # low temperature: we want as little creativity as possible
)

fixed_text = response.choices[0].message.content

Let's compare the two versions:

In [ ]:
print("=== BEFORE: raw Whisper transcription ===")
print(transcript_fr)
print()
print("=== AFTER: corrected by the LLM ===")
print(fixed_text)

🤔 **Question**: did the LLM respect rule 5 (never change the vocabulary)? Compare both versions carefully.

## 🎉 Well done!

You have automated the processing of audio recordings:
1. **automatic transcription** (Whisper) saves a huge amount of time compared to manual transcription,
2. an **LLM** can then clean the text or extract its themes, driven by a **system prompt**,
3. a **function** called in a loop processes a whole corpus.

**Key points for your own projects:**
- **Size vs performance**: we used small models (`whisper-1`, `gpt-4o-mini`): they already perform very well on these tasks.
- **Reproducibility**: scripting these steps makes the process transparent and repeatable.
- **Loss of authenticity**: cleaning by an LLM (and even the raw transcription) always risks changing the meaning, smoothing the language too much, or misunderstanding words. The AI does not replace your listening: always compare the text with the original audio.